# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}\n\nVersion: {metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their IDs.
<br>
Entities are referenced via their `@id` fields. Listing the record sets and available fields for inspection.

In [ ]:
from pprint import pprint

# List all record sets in the metadata with their @id
record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    record_sets = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs for rs in metadata.recordSet]
    print(f"Available Record Sets (@id):\n{record_sets}")
else:
    print('No record sets defined in metadata. Attempting to infer recordSet from data files...')
    # For demonstration, we attempt to read from each distribution, assuming each is a record set
    record_sets = [d['@id'] for d in metadata.distribution]
    print(f"Available Record Sets (@id):\n{record_sets}")

# Display fields/columns for each record set (using mlcroissant's schema parsing)
for record_set_id in record_sets:
    print(f"\n--- Fields for Record Set: {record_set_id} ---")
    try:
        records = dataset.records(record_set=record_set_id)
        sample = next(records)
        if isinstance(sample, dict):
            pprint(list(sample.keys()))
        else:
            print(type(sample))
    except Exception as e:
        print(f"Could not preview fields for {record_set_id}: {e}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
All entities are referenced by their `@id` fields.
Here, we'll load all available record sets.

In [ ]:
# Extract data from each record set
dataframes = {}

for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"\nColumns for Record Set {record_set_id}:")
            print(df.columns.tolist())
            print(f"\nPreview:")
            display(df.head())
        else:
            print(f"No records found for record set {record_set_id}.")
    except Exception as e:
        print(f"Failed to load records for {record_set_id}: {e}")

# For subsequent analysis, pick the first available dataframe
if dataframes:
    primary_record_set_id = next(iter(dataframes))
    primary_df = dataframes[primary_record_set_id]
else:
    primary_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
Operations include removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.
***All references use the `@id` fields for columns.***

In [ ]:
import numpy as np

# Example numeric field (referenced by @id): Use the first numeric column found
numeric_field_id = None
group_field_id = None

if primary_record_set_id:
    df = dataframes[primary_record_set_id]
    # Attempt to infer numeric fields
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        numeric_field_id = numeric_cols[0]
        print(f"Using numeric field: {numeric_field_id}")
    else:
        print("No numeric columns available in the sample record set.")
    # Attempt to infer a group field (categorical)
    cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    if cat_cols:
        group_field_id = cat_cols[0]
        print(f"Using group field: {group_field_id}")
    else:
        print("No categorical columns available for grouping.")

    # If a numeric field is available, perform filtering and normalization
    if numeric_field_id:
        threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by group_field if available
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
            display(grouped_df.head())
else:
    print("No data loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here, we create a simple histogram and boxplot of the numeric field (referenced by its `@id`). If a group field is present, show a group-wise boxplot.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if primary_record_set_id and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(7, 4))
    sns.boxplot(y=df[numeric_field_id].dropna())
    plt.title(f"Boxplot of {numeric_field_id}")
    plt.tight_layout()
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(12, 5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook demonstrated how to use `mlcroissant` to load the FAIR^2 dataset
by referencing record sets and fields exclusively by their `@id`.

- Loaded metadata and previewed dataset structure
- Extracted tabular data from available record sets
- Performed basic filtering, normalization, and grouping
- Visualized numeric data distributions

Further analysis can extend this template to domain-specific insights and modeling. For reproducible FAIR results, always reference entities using their `@id`.